# Analisis Pendapatan Transaksi Point of Sale (2024–2025)
**Mata Kuliah: Data Science**  
Firdaus — NIM 10124497

---

Dokumentasi pemrosesan dan analisis data transaksi Point of Sale (POS) untuk komparasi performa pendapatan kotor dan bersih antara tahun 2024 dan 2025.

### 1. Inisialisasi Dependensi
Memuat pustaka pandas dan numpy untuk manipulasi data tabular dan operasi komputasi numerik.

In [1]:
import pandas as pd
import numpy as np

### 2. Pemuatan Dataset
Membaca dataset transaksi dari file CSV ke dalam DataFrame dan meninjau struktur sampel awal.

In [2]:
file_name = 'data/paket_a_transaksi_pos_2024_2025.csv'
df = pd.read_csv(file_name)

df.head()

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


### 3. Pemeriksaan Kualitas Data
Identifikasi jumlah nilai kosong (*missing values*) pada setiap atribut data transaksi.

In [3]:
df.isnull().sum()

id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64

### 4. Pembersihan Data dan Imputasi
Imputasi nilai kosong pada atribut `kode_kasir` dengan label `Unknown` dan standardisasi tipe data numerik.

In [4]:
df_clean = df.copy()
df_clean['kode_kasir'] = df_clean['kode_kasir'].fillna('Unknown')

num_cols = ['jumlah', 'harga_satuan', 'diskon_persen']
for col in num_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

### 5. Filtrasi Transaksi Valid
Menyaring data transaksi berdasarkan status berhasil untuk memastikan akurasi perhitungan pendapatan.

In [5]:
status_valid = ['SELESAI', 'Selesai', 'SUCCESS', 'Success', 'Completed', 'Paid']
df_valid = df_clean[df_clean['status'].astype(str).str.strip().isin(status_valid)].copy()

df_valid.head()

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000.0,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000.0,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,0.0,0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500.0,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000.0,0,SELESAI


### 6. Ekstraksi Fitur dan Perhitungan Pendapatan
Mengekstrak atribut tahun dari tanggal transaksi serta menghitung pendapatan bersih setelah pemotongan diskon.

In [6]:
df_valid['tanggal_dt'] = pd.to_datetime(df_valid['tanggal'], format='mixed', dayfirst=True)
df_valid['tahun'] = df_valid['tanggal_dt'].dt.year

subtotal = df_valid['harga_satuan'] * df_valid['jumlah']
diskon_nominal = subtotal * (df_valid['diskon_persen'] / 100)
df_valid['pendapatan'] = subtotal - diskon_nominal

df_valid[['id_transaksi', 'tanggal_dt', 'tahun', 'jumlah', 'harga_satuan', 'diskon_persen', 'pendapatan']].head()

,id_transaksi,tanggal_dt,tahun,jumlah,harga_satuan,diskon_persen,pendapatan
0,TRX0241065,2025-02-06,2025,4,13000.0,0,52000.0
1,TRX0240510,2024-12-28,2024,1,21000.0,0,21000.0
2,TRX0244483,2024-04-09,2024,3,0.0,0,0.0
3,TRX0242532,2025-07-30,2025,2,6500.0,0,13000.0
4,TRX0243509,2025-03-26,2025,1,6000.0,0,6000.0


### 7. Agregasi Pendapatan Tahunan
Pemisahan data berdasarkan periode tahun 2024 dan 2025 serta akumulasi total pendapatan.

In [7]:
df_2024 = df_valid[df_valid['tahun'] == 2024].copy()
df_2025 = df_valid[df_valid['tahun'] == 2025].copy()

total_2024 = df_2024['pendapatan'].sum()
total_2025 = df_2025['pendapatan'].sum()

print(f"Total Pendapatan 2024 ({len(df_2024):,} transaksi): Rp {total_2024:,.2f}")
print(f"Total Pendapatan 2025 ({len(df_2025):,} transaksi): Rp {total_2025:,.2f}")

Total Pendapatan 2024 (2,364 transaksi): Rp 385,920,077.40
Total Pendapatan 2025 (2,385 transaksi): Rp 244,899,662.90


### 8. Ekspor Hasil Analisis
Penyimpanan ringkasan pendapatan tahunan dan dataset hasil pembersihan ke dalam file CSV.

In [8]:
df_summary = pd.DataFrame({
    'Tahun': [2024, 2025],
    'Jumlah_Data': [len(df_2024), len(df_2025)],
    'Total_Pendapatan': [total_2024, total_2025]
})

df_summary.to_csv('output/ringkasan_pendapatan_2024_2025.csv', index=False)
df_valid.to_csv('output/dataset_cleaned.csv', index=False)

df_summary

,Tahun,Jumlah_Data,Total_Pendapatan
0,2024,2364,385920077.4
1,2025,2385,244899662.9
